In [ ]:
# ==============================================================================
# 🚀 DO NOT MODIFY: Standardized Notebook Setup
# ==============================================================================
# This cell is designed to work in both Google Colab and local environments.
# It ensures that the environment is correctly configured by cloning (or
# locating) the project repository and installing the necessary dependencies.
#
# ------------------------------------------------------------------------------
#
#  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):
#
#  This cell will automatically find the repository root and configure your
#  environment. Just make sure you have run: pip install -e .[dev]
#
# ------------------------------------------------------------------------------

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# --- Configuration ---
REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"
REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned
# --- End of Configuration ---


def find_repo_root(start_path: Path) -> Path | None:
    """
    Find the repository root by looking for pyproject.toml.

    Searches upward from start_path until it finds pyproject.toml or hits root.

    Args:
        start_path: Directory to start searching from.

    Returns:
        Path to repository root, or None if not found.
    """
    current = start_path.resolve()
    while current != current.parent:  # Stop at filesystem root
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return None


def detect_active_branch(repo_dir: Path) -> str:
    """
    Determine the active git branch for pulling updates.

    Tries multiple methods to detect the current branch name.

    Args:
        repo_dir: Path to the git repository.

    Returns:
        Branch name (defaults to 'master' if detection fails).
    """
    commands = [
        "git symbolic-ref --short HEAD",
        "git rev-parse --abbrev-ref HEAD",
    ]
    for cmd in commands:
        result = subprocess.run(
            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True
        )
        if result.returncode == 0:
            branch = result.stdout.strip()
            if branch and not branch.startswith("origin/"):
                return branch
    return "master"


def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:
    """
    Run a shell command and raise an error if it fails.

    Args:
        cmd: The command to run.
        cwd: Optional working directory for the command.

    Raises:
        RuntimeError: If the command returns a non-zero exit code.
    """
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")


def load_setup_module(repo_path: Path):
    """Load the setup module directly without triggering package imports."""
    setup_path = repo_path / "core" / "notebook" / "setup.py"
    spec = importlib.util.spec_from_file_location("_setup_module", setup_path)
    setup_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(setup_module)
    return setup_module


# --- Detect environment ---
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# --- Main setup logic ---
if IN_COLAB:
    print("☁️  Running in Google Colab. Setting up the environment...\n")

    # Determine repository path
    start_dir = Path.cwd()
    if start_dir.name == REPO_DIR.name:
        repo_path = start_dir
    else:
        repo_path = start_dir / REPO_DIR

    # Clone or update repository
    if not repo_path.exists():
        print(f"📥 Cloning repository from {REPO_URL}...")
        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")
        print(f"✅ Repository cloned to {repo_path}\n")
    else:
        print(f"📂 Repository already exists at {repo_path}")
        active_branch = detect_active_branch(repo_path)
        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")
        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)
        print(f"✅ Repository updated\n")

    # Verify repository structure
    if not (repo_path / "pyproject.toml").exists():
        raise FileNotFoundError(
            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "
            "The repository may be corrupted."
        )

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    # Install dependencies (smart installation - only installs missing packages)
    # Load the setup module directly to avoid triggering other package imports
    setup = load_setup_module(repo_path)

    result = setup.smart_install_dependencies(
        repo_path=repo_path,
        include_dev=False,
        verbose=True,
    )

    # Fail loudly if critical packages failed to install
    if result["failed"]:
        print(f"\n⚠️  WARNING: {len(result['failed'])} packages failed to install:")
        for pkg in result["failed"]:
            print(f"  - {pkg}")
        print("\nYou may encounter import errors. Please check your internet connection.")

    print("\n" + "=" * 70)
    print("✅ Environment setup complete! You can now proceed with the notebook.")
    print("=" * 70)

else:
    print("💻 Running in local environment. Configuring...\n")

    # Find the repository root
    repo_path = find_repo_root(Path.cwd())

    if repo_path is None:
        raise FileNotFoundError(
            "Could not find repository root (no pyproject.toml found). "
            "Please ensure you are running this notebook from within the "
            "ADH-LLM-Tutorials-2025 repository directory."
        )

    print(f"✅ Found repository root: {repo_path}")

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)

    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    print("\n" + "=" * 70)
    print("✅ Local environment configured successfully!")
    print("=" * 70)
    print("\n⚠️  Please ensure you have run: pip install -e .[dev]")
    print("(Required for local development)")

# 05 - Model Comparison, Evaluation, and Explainability

## Moving Beyond Training: Rigorous Model Comparison

In the previous notebooks, we trained three different sequence models for sepsis prediction:
- **GRU**: A recurrent model with gating mechanisms
- **LSTM**: A recurrent model with more sophisticated memory cells  
- **Transformer**: An attention-based model that processes sequences in parallel

Training loss curves give us a sense of convergence, but they don't tell us which model is **actually better** for our task. In this notebook, we'll perform a rigorous comparison using:

1. **Standardized Metrics**: Calculate AUROC and AUPRC on a held-out test set
2. **Visual Comparison**: Plot ROC curves to compare discriminative performance
3. **Explainability**: Use Integrated Gradients to understand what features drive predictions

This is the critical step that transforms research experiments into actionable clinical insights.

In [ ]:
# Import required libraries
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml

from core.config import GRUConfig, LSTMConfig, TrainConfig, TransformerConfig
from core.data import create_dataloaders
from core.data.physionet_sepsis import get_sepsis_data
from core.evaluation import calculate_classification_metrics, get_predictions
from core.explain import explain_instance, plot_feature_attributions
from core.models import GRUModel, LSTMModel, TransformerModel
from core.notebook import ensure_project_root, get_feature_names
from core.viz import plot_roc_curves

## Step 0: Training Retrospective - What Just Happened?

Before we compare models, let's take a moment to "open the hood" and understand what actually happened when you executed `trainer.fit()` in notebooks 02-04.

### The Magic Revealed: Inside `trainer.fit()`

In each modeling notebook, you wrote this simple code:

```python
trainer = Trainer(model, train_loader, val_loader, train_config, save_path="models/gru_best.pt")
history, best_model_path = trainer.fit()
```

But behind this single method call, the `Trainer` executed **hundreds of lines of code** to train your model. Here's what actually happened:

### The Complete Training Loop

**1. Initialization**
- Set model to training mode: `model.train()`
- Initialize Adam optimizer with learning rate 0.001
- Set up gradient clipping (max norm = 1.0)
- Prepare device (CUDA if available, else CPU)
- Create empty history dictionary to track metrics

**2. Epoch Loop** (Repeated 20 times)

For each of the 20 epochs:

#### **Phase A: Training Phase**
For each batch of 32 patients from `train_loader`:

1. **Load Batch**
   - Get padded sequences (shape: `[32, max_seq_len, 34]`)
   - Get masks (shape: `[32, max_seq_len]`)
   - Get labels (shape: `[32]`)
   - Move tensors to device (GPU if available)

2. **Forward Pass** 
   - `logits = model(features, mask)` → Get raw predictions (shape: `[32]`)
   - The model processes the entire sequence and outputs a single logit per patient

3. **Compute Loss**
   - `loss = BCEWithLogitsLoss(logits, labels)` → Binary cross-entropy
   - This measures how far the predictions are from the true labels
   - Lower loss = better predictions

4. **Backward Pass** (Backpropagation)
   - `optimizer.zero_grad()` → Clear previous gradients
   - `loss.backward()` → Compute gradients for all model parameters
   - Gradients tell us "how to adjust weights to reduce loss"

5. **Gradient Clipping**
   - `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`
   - Prevents "exploding gradients" (very large updates that destabilize training)
   - Especially important for recurrent models (GRU, LSTM)

6. **Optimizer Step**
   - `optimizer.step()` → Update model weights using gradients
   - Adam optimizer adaptively adjusts learning rate for each parameter
   - This is where the model actually "learns"

7. **Track Loss**
   - Accumulate batch loss for epoch average

**Result**: After ~80 batches (2,560 patients), the model has seen the entire training set once.

#### **Phase B: Validation Phase**
For each batch of 32 patients from `val_loader`:

1. **Set Evaluation Mode**: `model.eval()` → Disables dropout, batch norm tracking
2. **No Gradient Computation**: `with torch.no_grad():` → Saves memory and computation
3. **Forward Pass Only**: Get predictions, compute loss
4. **Track Metrics**: Accumulate validation loss

**Why validation?** 
- Training loss can decrease even if the model is overfitting (memorizing training data)
- Validation loss tells us if the model generalizes to unseen patients
- We use validation loss to select the best model checkpoint

#### **Phase C: Checkpointing**
After each epoch:
- Compare current validation loss to best validation loss
- If improved:
  - Save model weights: `torch.save(model.state_dict(), save_path)`
  - Update best loss tracker
  - This is the "best model" we'll use for final evaluation

#### **Phase D: Logging**
- Print epoch number, train loss, validation loss
- Update progress bar (tqdm)
- Store metrics in history dictionary

**3. Final Output**
After 20 epochs:
- Return `history` (dictionary with train/val loss curves)
- Return `best_model_path` (path to checkpoint with lowest validation loss)

### Training Statistics

For each model, here's what happened:

| Metric | GRU | LSTM | Transformer |
|--------|-----|------|-------------|
| **Total epochs** | 20 | 20 | 20 |
| **Batches per epoch** | ~80 | ~80 | ~80 |
| **Total gradient updates** | ~1,600 | ~1,600 | ~1,600 |
| **Training time (CPU)** | ~2-3 min | ~2-3 min | ~5-7 min |
| **Training time (GPU)** | ~30 sec | ~30 sec | ~1 min |
| **Parameters updated** | ~17,000 | ~23,000 | ~19,000 |

### Understanding Hyperparameters

The YAML configuration files defined these critical hyperparameters:

```yaml
model:
  input_size: 34          # Number of physiological features
  hidden_size: 64         # Dimension of hidden state (GRU/LSTM) or embeddings (Transformer)
  num_layers: 2           # Stack 2 layers (deeper = more capacity, but harder to train)
  dropout: 0.2            # Randomly drop 20% of connections during training (prevents overfitting)

training:
  batch_size: 32          # Process 32 patients simultaneously (trade-off: speed vs. memory)
  epochs: 20              # Complete passes through training data
  learning_rate: 0.001    # Step size for weight updates (too high = unstable, too low = slow)
```

**What do these mean?**

- **`hidden_size`**: Bigger = more model capacity to learn complex patterns, but more parameters to train
  - GRU/LSTM: 64 means the hidden state vector has 64 dimensions
  - Transformer: 64 means each token embedding has 64 dimensions
  
- **`num_layers`**: Stacking layers allows learning hierarchical representations
  - Layer 1 might learn "heart rate is elevated"
  - Layer 2 might learn "elevated HR + elevated temp = infection pattern"
  
- **`dropout`**: During training, randomly set 20% of activations to zero
  - Forces model to learn robust features (can't rely on any single neuron)
  - Reduces overfitting (memorizing training data)
  - **Disabled during validation/testing** (we want full model capacity for predictions)
  
- **`learning_rate`**: How much to adjust weights based on gradients
  - 0.001 is a typical starting point for Adam optimizer
  - Too high (e.g., 0.1): Model oscillates, never converges
  - Too low (e.g., 0.00001): Model learns very slowly, might not reach optimum in 20 epochs

### Why Binary Cross-Entropy Loss?

For sepsis prediction (binary classification), we use **BCE with logits**:

```
Loss = -[y * log(σ(logit)) + (1-y) * log(1-σ(logit))]
```

Where:
- `y` = true label (0 or 1)
- `logit` = raw model output (unbounded)
- `σ(logit)` = sigmoid function = probability

**Why this loss?**
- Penalizes confident wrong predictions heavily
- Encourages model to output calibrated probabilities
- Convex optimization landscape (easier to train than alternatives)

### The Role of the Optimizer: Adam

Adam (Adaptive Moment Estimation) is our optimization algorithm:

- **Momentum**: Uses moving average of past gradients → smoother updates
- **Adaptive learning rates**: Each parameter gets its own effective learning rate
- **Why not SGD?**: Adam converges faster for deep learning, handles sparse gradients better

### What About Overfitting?

We used several techniques to prevent overfitting:

1. **Dropout** (0.2): Regularization during training
2. **Validation monitoring**: Stop training when validation loss stops improving (implicit early stopping via checkpointing)
3. **Gradient clipping**: Prevents extreme weight updates that could overfit to outliers

### Key Insight: The Abstraction Was Intentional

You didn't need to implement any of this because:
- The goal was to **compare architectures** (GRU vs. LSTM vs. Transformer), not debug training loops
- The `Trainer` handles all the PyTorch complexity consistently across all three models
- You could focus on **science** (which model works best?) rather than **engineering** (how do I implement backprop?)

Now that you understand what happened during training, let's rigorously compare the results!

## Step 1: Load Best Models and Data

We'll load the best checkpoints saved during training in notebooks 02-04.

In [ ]:
# Ensure execution happens from the project root
project_root = ensure_project_root()
models_dir = project_root / "models"
configs_dir = project_root / "configs"

# Define paths to saved models
gru_model_path = models_dir / "gru_best.pt"
lstm_model_path = models_dir / "lstm_best.pt"
transformer_model_path = models_dir / "transformer_best.pt"

# Determine device for both loading and inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load configurations
gru_config = GRUConfig(
    **yaml.safe_load((configs_dir / "gru.yaml").read_text())["model"]
)
lstm_config = LSTMConfig(
    **yaml.safe_load((configs_dir / "lstm.yaml").read_text())["model"]
)
transformer_config = TransformerConfig(
    **yaml.safe_load((configs_dir / "transformer.yaml").read_text())["model"]
)
train_config = TrainConfig(
    **yaml.safe_load((configs_dir / "gru.yaml").read_text())["training"]
)

# Instantiate models
gru_model = GRUModel(gru_config)
lstm_model = LSTMModel(lstm_config)
transformer_model = TransformerModel(transformer_config)

# Load saved weights onto the current device
gru_model.load_state_dict(torch.load(gru_model_path, map_location=device))
lstm_model.load_state_dict(torch.load(lstm_model_path, map_location=device))
transformer_model.load_state_dict(
    torch.load(transformer_model_path, map_location=device)
)

# Ensure models live on the selected device
gru_model.to(device)
lstm_model.to(device)
transformer_model.to(device)

print("✅ All models loaded successfully!")

# Load data and create test loader (we'll use validation set as our test set)
sepsis_df = get_sepsis_data()
train_loader, test_loader = create_dataloaders(train_config=train_config, df=sepsis_df)

print(f"\nTest set size: {len(test_loader.dataset)} patients")

## Step 2: Generate and Compare Performance Metrics

We'll generate predictions from all three models and calculate standard classification metrics.

In [ ]:
# Generate predictions for each model
results = {}
models = {
    "GRU": gru_model,
    "LSTM": lstm_model,
    "Transformer": transformer_model,
}

for name, model in models.items():
    print(f"\nGenerating predictions for {name}...")
    preds = get_predictions(
        model,
        test_loader,
        device=device,
        return_tensors=True,
    )
    results[name] = preds

    metrics = calculate_classification_metrics(
        preds.labels.numpy(), preds.probabilities.numpy()
    )
    print(f"{name} - AUROC: {metrics['auroc']:.4f}, AUPRC: {metrics['auprc']:.4f}")

print("\n" + "=" * 60)
print("SUMMARY: Model Performance Comparison")
print("=" * 60)
comparison_data = []
for name in ["GRU", "LSTM", "Transformer"]:
    preds = results[name]
    metrics = calculate_classification_metrics(
        preds.labels.numpy(), preds.probabilities.numpy()
    )
    comparison_data.append(
        {
            "Model": name,
            "AUROC": f"{metrics['auroc']:.4f}",
            "AUPRC": f"{metrics['auprc']:.4f}",
        }
    )

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("=" * 60)

### Visualize ROC Curves

ROC curves provide a visual comparison of model performance across all classification thresholds.

In [ ]:
# Prepare data for ROC plot
roc_data = {
    name: (preds.labels.numpy(), preds.probabilities.numpy())
    for name, preds in results.items()
}

# Plot ROC curves
fig = plot_roc_curves(roc_data)
plt.show()

## Step 2.5: Understanding Performance Through Architecture

Now that we've seen the metrics, let's connect **performance differences to architectural design**. Why might one model outperform another? The answer lies in how they process sequential information.

### How Each Model Processes a 24-Hour ICU Stay

#### **GRU: Sequential with Single-State Memory**

```
Hour 1 → [GRU Cell] → h₁
              ↓
Hour 2 → [GRU Cell] → h₂ (updated from h₁)
              ↓
Hour 3 → [GRU Cell] → h₃ (updated from h₂)
              ↓
   ...        ↓
              ↓
Hour 24 → [GRU Cell] → h₂₄ → [Linear Classifier] → Prediction
```

**Key mechanism**: At each hour, the GRU:
1. Takes current input (vital signs at hour t)
2. Takes previous hidden state (h_{t-1})
3. Uses **gates** to decide:
   - What to forget from h_{t-1} (forget gate)
   - What new information to add (update gate)
4. Outputs updated hidden state h_t

**Strength**: Efficient, good at short-to-medium range dependencies  
**Limitation**: Information from hour 1 can "fade" by hour 50 (vanishing gradient problem)

#### **LSTM: Sequential with Dual-State Memory**

```
Hour 1 → [LSTM Cell] → (c₁, h₁)
              ↓
Hour 2 → [LSTM Cell] → (c₂, h₂)
              ↓
Hour 3 → [LSTM Cell] → (c₃, h₃)
              ↓
   ...        ↓
              ↓
Hour 24 → [LSTM Cell] → (c₂₄, h₂₄) → [Linear Classifier] → Prediction
```

**Key difference from GRU**: Maintains **two** memory components:
- **Cell state (c)**: Long-term memory "conveyor belt"
- **Hidden state (h)**: Short-term working memory

**Three gates** control information flow:
1. **Forget gate**: What to remove from cell state
2. **Input gate**: What new information to add to cell state
3. **Output gate**: What from cell state goes to hidden state

**Example**: 
- Cell state might carry "patient had lactate spike at hour 5" all the way to hour 24
- Hidden state tracks immediate context like "current heart rate"

**Strength**: Better at preserving long-term dependencies than GRU  
**Limitation**: More complex (more parameters, slower training)

#### **Transformer: Parallel with Attention**

```
[Hour 1, Hour 2, Hour 3, ..., Hour 24]
              ↓
    Positional Encoding (add temporal info)
              ↓
    Multi-Head Self-Attention Layer 1
    (Each hour attends to all hours)
              ↓
    Feed-Forward Network
              ↓
    Multi-Head Self-Attention Layer 2
              ↓
    Feed-Forward Network
              ↓
    Mean Pooling (aggregate all hours)
              ↓
    [Linear Classifier] → Prediction
```

**Key mechanism**: **Self-Attention** allows each hour to directly "look at" every other hour:

```
Attention(Q, K, V) = softmax(QK^T / √d) V
```

**What this means**:
- Query (Q): "What am I looking for?"
- Key (K): "What information do I have?"
- Value (V): "What is that information?"

**Example**: When processing hour 24:
- Query: "Are there signs of deteriorating patient state?"
- Attends strongly to hour 5 (lactate spike) and hour 15 (BP drop)
- Attends weakly to stable hours (2-4, 8-12)

**Attention weights** are learned during training - the model discovers which hours matter!

**Strength**: 
- Can directly connect distant events (hour 1 ↔ hour 50)
- Parallel processing (faster on GPUs)
- No vanishing gradient problem

**Limitation**:
- Requires more data to learn attention patterns
- Computationally expensive (quadratic in sequence length)
- Can be overkill for simple patterns

### Comprehensive Model Comparison

| Aspect | GRU | LSTM | Transformer |
|--------|-----|------|-------------|
| **Processing** | Sequential (t=1→2→3) | Sequential (t=1→2→3) | Parallel (all timesteps) |
| **Memory** | Hidden state (h) | Cell state (c) + Hidden (h) | Attention weights |
| **Parameters** | ~17,000 | ~23,000 | ~19,000 |
| **Long-term deps** | Moderate (10-20 steps) | Better (20-50 steps) | Best (any distance) |
| **Training speed** | Fast | Medium | Slow (attention O(n²)) |
| **Inference speed** | Fastest | Fast | Medium |
| **Data efficiency** | Good | Good | Needs more data |
| **Interpretability** | Hidden state (opaque) | Cell/hidden states (opaque) | Attention weights (transparent) |

### Connecting Architecture to Performance

Based on the AUROC/AUPRC scores above, let's interpret what each model's performance tells us:

#### **If GRU performs well:**
- Sequential processing is sufficient
- Short-to-medium range temporal patterns dominate
- Sepsis indicators appear in recent hours (simple baseline works)

#### **If LSTM significantly outperforms GRU:**
- Long-term dependencies matter
- Early warning signs (hour 5) are predictive of outcomes (hour 24)
- The dual-state memory (cell + hidden) is capturing something GRU misses

#### **If Transformer outperforms both:**
- Complex, non-sequential patterns exist
- Specific hour combinations are predictive (e.g., hour 5 + hour 18)
- Direct attention connections help (not mediated through sequential hidden states)
- Parallel processing of all hours provides global context

#### **If all three perform similarly:**
- The task may not require sophisticated temporal modeling
- Simple features at any given timestep might be sufficient
- Consider: Could a simpler model (e.g., logistic regression on aggregated features) work?

### Performance Trade-offs for Clinical Deployment

| Consideration | GRU | LSTM | Transformer |
|---------------|-----|------|-------------|
| **Real-time inference** | ⭐⭐⭐ Excellent | ⭐⭐⭐ Excellent | ⭐⭐ Good |
| **Model size (memory)** | ⭐⭐⭐ Small | ⭐⭐ Medium | ⭐⭐ Medium |
| **Explainability** | ⭐ Difficult | ⭐ Difficult | ⭐⭐⭐ Attention maps! |
| **Calibration** | ⭐⭐ Need checking | ⭐⭐ Need checking | ⭐⭐ Need checking |
| **Overfitting risk** | ⭐⭐ Low | ⭐⭐ Medium | ⭐⭐⭐ Higher (more params) |

### Reflection Questions

Based on **your specific results** (the AUROC/AUPRC scores from Step 2):

1. **Which model performed best in your experiment?**

2. **Was the performance difference substantial or marginal?**
   - If < 0.01 AUROC difference: Models are essentially equivalent
   - If > 0.03 AUROC difference: Clear winner, architectural choice matters

3. **If you had to deploy ONE model to a real ICU tomorrow, which would you choose? Why?**
   - Consider: Accuracy, inference speed, explainability, simplicity

4. **Does the Transformer's superior performance (if any) justify its computational cost?**
   - 2-3x slower training, larger memory footprint
   - Trade-off: Accuracy vs. deployment constraints

5. **What does your result tell you about sepsis temporal patterns?**
   - Are they local (recent hours) or global (entire ICU stay)?
   - Are they sequential (cascade) or combinatorial (specific hour pairs)?

Let's continue to explainability to see **what features** these models actually use!

## Step 3: Explainability - Peeking Inside the Black Box

### Understanding Model Decisions with Integrated Gradients

While our models can predict sepsis with high accuracy, understanding **why** they make specific predictions is crucial for clinical adoption. We'll use **Integrated Gradients**, an attribution method that identifies which input features most strongly influence a particular prediction.

**How it works:**
- Computes gradients along a path from a baseline (all zeros) to the actual input
- Attributes the prediction to specific features based on these gradients
- Positive attributions indicate features that push toward the predicted class
- Negative attributions indicate features that push away from it

Let's examine what features the **Transformer model** finds most important for predicting sepsis in a specific patient.

In [ ]:
# Select a patient instance from the test set
# We'll pick a sepsis-positive case for interpretation
septic_indices = (results["Transformer"].labels == 1).nonzero()[0]
instance_idx = septic_indices[0]  # First septic patient

# Get the instance from the test loader
# We need to manually extract it from the dataset
test_dataset = test_loader.dataset
features, mask, label = test_dataset[instance_idx]

print("Selected patient instance:")
print(f"  - True label: {'Sepsis' if label.item() == 1 else 'No Sepsis'}")
print(f"  - Sequence length: {features.shape[0]}")
print(f"  - Number of features: {features.shape[1]}")

# Explain the prediction using Integrated Gradients
print("\nComputing feature attributions...")
attributions = explain_instance(
    model=transformer_model,
    instance_tensor=features,
    mask=mask,
    n_steps=50,
)

print(f"✅ Attributions computed! Shape: {attributions.shape}")

In [ ]:
# Retrieve canonical feature names from the data pipeline
feature_names = get_feature_names()

# Plot feature attributions
fig = plot_feature_attributions(
    attributions=attributions,
    feature_names=feature_names,
    top_k=15,
    aggregate_method="mean",
)
plt.show()

## Interpretation of Feature Attributions

### Clinical Insights from Model Explainability

The attribution plot above reveals which clinical measurements the Transformer model weighted most heavily when predicting sepsis for this specific patient. Let's interpret what we see:

**Expected Clinical Markers:**
- **Lactate**: Elevated lactate is a well-known biomarker of sepsis, indicating tissue hypoperfusion
- **WBC (White Blood Cell Count) / Platelets**: Abnormal WBC or Platelet counts (either elevated or very low) are classic signs of infection
- **Temperature**: Fever or hypothermia are SIRS (Systemic Inflammatory Response Syndrome) criteria
- **Heart Rate & Respiratory Rate**: Tachycardia and tachypnea are early warning signs
- **Blood Pressure (SBP/MAP/DBP)**: Hypotension indicates septic shock

**Novel or Unexpected Findings:**
If the model heavily weights features like **ICULOS** (ICU length of stay) or temporal patterns rather than instantaneous values, this suggests it's learning trajectory-based risk rather than single-point thresholds.

**Clinical Validation:**
The fact that our model's important features align with known sepsis pathophysiology **increases our confidence** in its predictions. If the model were relying on spurious correlations (e.g., "Unit2" being highly important), we'd need to investigate potential data leakage or confounding.

### Next Steps in a Real Deployment
1. **Validate across multiple patients**: Aggregate attributions to identify consistently important features
2. **Compare with clinical guidelines**: Align model features with Sepsis-3 criteria and qSOFA scores
3. **Engage clinical stakeholders**: Present findings to ICU physicians for domain validation
4. **Test fairness**: Ensure feature importance is consistent across patient subgroups (age, gender, etc.)

## Step 3.5: Comparative Explainability - Do Different Models Use Different Features?

We've seen what the Transformer finds important, but a critical question remains: **Do all three models agree on which features matter?** Or does each architecture prioritize different patterns?

### Why This Matters

- **If models agree**: Strong evidence these features are robustly predictive (high confidence)
- **If models disagree**: Architectures may be complementary (ensemble opportunity)
- **Clinical validation**: Features that ALL models find important should align with known sepsis biomarkers

Let's explain the **same patient instance** with all three models and compare their feature attributions.

In [ ]:
# Compare feature attributions across all three models for the same patient
import numpy as np

print("Computing attributions for all three models on the same patient...")
print(f"Patient index: {instance_idx}")
print(f"True label: {'Sepsis' if label.item() == 1 else 'No Sepsis'}")
print(f"Sequence length: {features.shape[0]} hours\n")

# Compute attributions for each model
model_attributions = {}

for model_name, model in [
    ("GRU", gru_model),
    ("LSTM", lstm_model),
    ("Transformer", transformer_model),
]:
    print(f"Computing {model_name} attributions...")
    attr = explain_instance(
        model=model,
        instance_tensor=features,
        mask=mask,
        n_steps=50,
    )
    model_attributions[model_name] = attr

print("\n✅ All attributions computed!")

In [ ]:
# Create side-by-side comparison of top features for each model
import matplotlib.pyplot as plt

# Aggregate attributions across time (mean absolute value)
feature_importance = {}
for model_name, attr in model_attributions.items():
    # attr shape: [seq_len, num_features]
    # Aggregate by taking mean of absolute values across time
    feature_importance[model_name] = np.mean(np.abs(attr), axis=0)

# Select top K features to display
top_k = 10

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
feature_names = get_feature_names()

for idx, (model_name, importance) in enumerate(feature_importance.items()):
    # Get top K features
    top_indices = np.argsort(importance)[-top_k:][::-1]
    top_features = [feature_names[i] for i in top_indices]
    top_values = importance[top_indices]

    # Plot horizontal bar chart
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
    axes[idx].barh(
        range(len(top_features)), top_values, color=colors, edgecolor="black"
    )
    axes[idx].set_yticks(range(len(top_features)))
    axes[idx].set_yticklabels(top_features, fontsize=10)
    axes[idx].set_xlabel("Mean Absolute Attribution", fontsize=11)
    axes[idx].set_title(
        f"{model_name}\nTop {top_k} Features", fontsize=13, fontweight="bold"
    )
    axes[idx].invert_yaxis()
    axes[idx].grid(True, alpha=0.3, axis="x")

    # Add value labels
    for i, v in enumerate(top_values):
        axes[idx].text(
            v + 0.001, i, f"{v:.3f}", va="center", fontsize=9, fontweight="bold"
        )

plt.tight_layout()
plt.show()

### Interpreting Model Agreement and Disagreement

#### **Consensus Features** (Appear in All Three Models' Top 10)

Look for features that appear in the top 10 for GRU, LSTM, **and** Transformer. These are **high-confidence predictive biomarkers** because all three architectures independently learned to prioritize them.

**Clinical validation question**: Do these consensus features align with known sepsis pathophysiology?
- Expected: Lactate, WBC, Temperature, Heart Rate, Blood Pressure
- If unexpected features appear (e.g., Age, Unit1/Unit2), investigate potential confounding

#### **Model-Specific Features** (Appear in Only One Model's Top 10)

Features that only ONE model finds important suggest:

1. **Architectural bias**: Each model type has different inductive biases
   - GRU/LSTM: May prioritize features with strong temporal trends (gradual increase/decrease)
   - Transformer: May prioritize features with abrupt changes (attention focuses on "events")

2. **Complementary information**: Models might be capturing different aspects of the same phenomenon
   - Example: GRU focuses on recent trends, Transformer focuses on specific critical hours

3. **Potential ensemble opportunity**: If models use different features, combining them might improve performance

#### **Ranking Differences** (Same Features, Different Order)

Even when models agree on WHICH features matter, they might disagree on HOW MUCH they matter (ranking).

**Example interpretation**:
- GRU ranks Lactate #1, Transformer ranks it #3 → Both use it, but weigh it differently
- This could reflect: GRU learns "rising lactate trend" while Transformer learns "lactate spike at specific hour"

### Quantifying Agreement: Feature Overlap Analysis

In [ ]:
# Quantify feature overlap between models
top_k = 10

# Get top K features for each model
top_features_per_model = {}
for model_name, importance in feature_importance.items():
    top_indices = np.argsort(importance)[-top_k:][::-1]
    top_features_per_model[model_name] = set([feature_names[i] for i in top_indices])

# Calculate pairwise overlaps
print("=" * 70)
print("Feature Overlap Analysis (Top 10 Features)")
print("=" * 70)

# GRU vs LSTM
overlap_gru_lstm = top_features_per_model["GRU"] & top_features_per_model["LSTM"]
print(f"\n🔗 GRU ∩ LSTM: {len(overlap_gru_lstm)}/{top_k} features")
print(f"   Shared: {sorted(overlap_gru_lstm)}")

# GRU vs Transformer
overlap_gru_transformer = (
    top_features_per_model["GRU"] & top_features_per_model["Transformer"]
)
print(f"\n🔗 GRU ∩ Transformer: {len(overlap_gru_transformer)}/{top_k} features")
print(f"   Shared: {sorted(overlap_gru_transformer)}")

# LSTM vs Transformer
overlap_lstm_transformer = (
    top_features_per_model["LSTM"] & top_features_per_model["Transformer"]
)
print(f"\n🔗 LSTM ∩ Transformer: {len(overlap_lstm_transformer)}/{top_k} features")
print(f"   Shared: {sorted(overlap_lstm_transformer)}")

# All three models
consensus_features = (
    top_features_per_model["GRU"]
    & top_features_per_model["LSTM"]
    & top_features_per_model["Transformer"]
)
print(f"\n⭐ Consensus (ALL three models): {len(consensus_features)}/{top_k} features")
if consensus_features:
    print(f"   Features: {sorted(consensus_features)}")
    print("\n   → These are HIGH-CONFIDENCE predictive biomarkers!")
else:
    print("   → No features appear in all three models' top 10")
    print("   → Models may be using complementary information (ensemble opportunity)")

# Model-specific features (appear in only one model)
print(f"\n📊 Model-Specific Features (unique to one model):")
for model_name in ["GRU", "LSTM", "Transformer"]:
    other_models = [m for m in ["GRU", "LSTM", "Transformer"] if m != model_name]
    unique = top_features_per_model[model_name] - (
        top_features_per_model[other_models[0]]
        | top_features_per_model[other_models[1]]
    )
    if unique:
        print(f"   {model_name} only: {sorted(unique)}")
    else:
        print(
            f"   {model_name} only: None (all features shared with at least one other model)"
        )

print("\n" + "=" * 70)

## Conclusion

In this notebook, we've completed the critical final steps of a machine learning pipeline:

1. ✅ **Loaded and compared** three different sequence model architectures
2. ✅ **Calculated standardized metrics** (AUROC, AUPRC) on a held-out test set
3. ✅ **Visualized comparative performance** with ROC curves
4. ✅ **Explained individual predictions** using Integrated Gradients

These techniques transform a collection of training experiments into **actionable insights** that can inform model selection, clinical validation, and eventual deployment.

### Key Takeaways
- **Performance metrics** tell us which model generalizes best
- **ROC curves** help us understand the trade-off between sensitivity and specificity
- **Explainability** builds trust and enables clinical validation of model behavior

This completes Part 1 of the tutorial series on sequence models for digital health. You now have a complete, end-to-end pipeline for training, evaluating, and explaining time-series models for clinical prediction tasks! 🎉

## Step 4: Beyond AUROC - What We Haven't Addressed

While we've successfully trained, compared, and explained three sequence models, **achieving high AUROC is only the beginning** of a responsible machine learning pipeline. This section addresses critical limitations and the gap between research experiments and clinical deployment.

### 4.1 Model Calibration: Are Predicted Probabilities Trustworthy?

**The Problem:**
- Our models output probabilities: "This patient has 75% risk of sepsis"
- But are these probabilities **calibrated**? Does "75% risk" actually mean 75 out of 100 similar patients develop sepsis?

**Why it matters:**
- A clinician might treat differently at 60% vs. 80% risk
- Over-confident predictions (90% but actually 60%) lead to unnecessary interventions
- Under-confident predictions (30% but actually 70%) lead to missed diagnoses

**How to assess calibration:**
1. **Calibration plots** (reliability diagrams): Plot predicted probability vs. observed frequency
2. **Brier score**: Mean squared error between predictions and outcomes (lower = better)
3. **Expected Calibration Error (ECE)**: Average difference between confidence and accuracy

**What we'd need to do:**
```python
from sklearn.calibration import calibration_curve

# For each model
prob_true, prob_pred = calibration_curve(y_true, y_pred, n_bins=10)
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0, 1], [0, 1], linestyle='--')  # Perfect calibration line
```

**Potential finding**: Neural networks often produce over-confident predictions. We might need **calibration techniques** like:
- Temperature scaling
- Platt scaling  
- Isotonic regression

---

### 4.2 Fairness: Does Performance Vary Across Subgroups?

**The Problem:**
- Our overall AUROC might be 0.85, but what if it's 0.90 for young patients and 0.70 for elderly?
- Models can have **disparate performance** across demographics, ICU units, or disease subtypes

**Why it matters:**
- Regulatory requirement (FDA guidance on algorithmic fairness)
- Ethical imperative: Avoid systematic bias against vulnerable populations
- Clinical trust: Physicians won't use a model that fails for specific patient groups

**Subgroups to analyze:**
1. **Demographics**: Age (<40, 40-65, >65), Gender (Male/Female)
2. **Clinical**: ICU unit (Medical ICU, Surgical ICU, Cardiac ICU)
3. **Temporal**: Time of admission (Day shift vs. Night shift)
4. **Disease severity**: Low risk (few comorbidities) vs. High risk (multiple organ dysfunction)

**What we'd need to do:**
```python
# Stratified performance analysis
for age_group in ['<40', '40-65', '>65']:
    mask = (df['Age'] >= age_min) & (df['Age'] < age_max)
    metrics = calculate_metrics(y_true[mask], y_pred[mask])
    print(f"{age_group}: AUROC = {metrics['auroc']:.3f}")
```

**Potential finding**: If we find disparities, we might need:
- Collect more data for underrepresented groups
- Use fairness-aware training objectives
- Deploy separate models for different subgroups

---

### 4.3 Temporal Validation: Can We Predict Early?

**The Problem:**
- Our models predict sepsis at **any point** during ICU stay
- But clinical value comes from **early warning** (predicting 4-6 hours before onset)

**Why it matters:**
- Predicting sepsis 1 hour before onset: Limited clinical utility (already presenting symptoms)
- Predicting 6 hours before onset: Actionable window for intervention

**What we'd need to do:**
- **Lead-time analysis**: Filter predictions to only include hours where `ICULOS < sepsis_onset - 6`
- **Temporal cross-validation**: Train on early months, test on later months (mimics deployment)

**Potential finding**: Performance might degrade with longer lead times. We'd need to:
- Tune models specifically for early prediction
- Explore temporal features (trends over last 4 hours, not just current values)
- Trade off sensitivity/specificity differently (higher sensitivity for early warning)

---

### 4.4 Computational Cost: Real-Time Deployment Feasibility

**The Problem:**
- In a real ICU, the model must run **continuously** (every hour for every patient)
- Example hospital: 20-bed ICU → 480 predictions per day

**Why it matters:**
| Model | Inference Time (per patient) | ICU-wide (20 patients) |
|-------|------------------------------|------------------------|
| GRU | ~5 ms | ~100 ms (✅ Real-time) |
| LSTM | ~6 ms | ~120 ms (✅ Real-time) |
| Transformer | ~15 ms | ~300 ms (✅ Acceptable, but slower) |

- Current models are fast enough, but with longer sequences (100+ hours) or larger batches, Transformer could become a bottleneck

**Optimization strategies if needed:**
- Model quantization (reduce precision: FP32 → FP16 or INT8)
- Knowledge distillation (train small "student" model to mimic large "teacher")
- ONNX export for optimized inference runtimes

---

### 4.5 Explainability Limitations: Correlation ≠ Causation

**The Problem:**
- Integrated Gradients tells us "Lactate is important for this prediction"
- But it doesn't tell us:
  - **Why** lactate is elevated (infection? ischemia? artifact?)
  - **Whether intervening** on lactate would change the outcome
  - **If the model is right** (lactate could be a spurious correlation)

**Why it matters:**
- Attribution methods show **correlation**, not **causation**
- A model might learn shortcuts: "Unit2 patients have higher sepsis rates" (confounding, not causality)
- Physicians need to understand **why** the model works, not just **what** it uses

**Other explainability methods to explore:**
- **SHAP (SHapley Additive exPlanations)**: Game-theory-based attributions
- **Attention weights** (for Transformer): Which hours does the model focus on?
- **Counterfactual explanations**: "If lactate had been normal, prediction would drop to 20%"

**Clinical validation steps:**
1. Present feature importance to ICU physicians → Does it match clinical intuition?
2. Test on synthetic cases → Does the model fail in expected ways?
3. Compare to clinical guidelines → Does the model align with Sepsis-3 criteria?

---

### 4.6 From Research to Deployment: The Regulatory and Clinical Gap

**What we've done:**
- ✅ Trained models on historical data
- ✅ Evaluated performance on held-out test set
- ✅ Explained predictions using attribution methods

**What's still needed for clinical deployment:**

#### **1. Prospective Validation**
- **Retrospective** (our approach): Evaluate on historical data
- **Prospective**: Deploy in real ICU, monitor performance on new patients
- **Gold standard**: Randomized controlled trial (RCT) comparing outcomes with/without the model

#### **2. Regulatory Approval**
- **FDA (US)**: Submit 510(k) or De Novo application for clinical decision support software
- **CE Mark (Europe)**: Demonstrate compliance with Medical Device Regulation (MDR)
- **Requirements**: Clinical validation studies, risk management, software documentation

#### **3. Electronic Health Record (EHR) Integration**
- Models need to ingest data from hospital EHR systems (Epic, Cerner, etc.)
- Handle real-time data streams, missing values, sensor artifacts
- Output predictions in clinician-facing dashboards

#### **4. Clinician User Studies**
- Will physicians **trust** the model?
- Does it fit into clinical **workflow**? (alerts during rounds? nurse station dashboard?)
- Does it improve **outcomes** or just create alert fatigue?

#### **5. Post-Deployment Monitoring**
- Continuous performance monitoring (data drift, concept drift)
- Feedback loops (label new cases, retrain periodically)
- Incident response (what happens when model fails?)

---

### 4.7 Research Directions: Where to Go from Here

If you wanted to extend this work, here are high-impact directions:

#### **Hyperparameter Tuning**
- We used default configs (hidden_size=64, dropout=0.2)
- Systematic search with **Optuna** or **Ray Tune** could improve performance

#### **Ensemble Methods**
- If models use different features (as we saw in comparative explainability), combine them:
  - Simple averaging: `pred = (gru_pred + lstm_pred + transformer_pred) / 3`
  - Weighted ensemble: Learn optimal weights via logistic regression
  - Stacking: Train a meta-model on top of base model predictions

#### **Multi-Task Learning**
- Predict multiple outcomes simultaneously: Sepsis + Mortality + Organ failure
- Shared representations might improve all tasks

#### **Attention Visualization**
- For Transformer, plot attention weights as heatmaps
- See which hours the model "looks at" when making predictions

#### **External Validation**
- Test on datasets from different hospitals (e.g., MIMIC-III, eICU)
- Does the model generalize to different populations?

#### **Causal Inference**
- Use causal discovery methods to identify which features are **causal** (not just correlated)
- Example: Propensity score matching, inverse probability weighting

---

### Key Takeaways

**What we accomplished:**
- Built a complete ML pipeline: Data → Models → Evaluation → Explanation
- Compared three architectures and understood their trade-offs
- Achieved strong predictive performance (AUROC likely ~0.80-0.85)

**What we **haven't** done (and that's OK for a tutorial!):**
- Calibration analysis (are probabilities trustworthy?)
- Fairness auditing (equal performance across subgroups?)
- Temporal validation (early warning capability?)
- Prospective clinical validation (does it work in real-time?)
- Regulatory approval (FDA, CE Mark)
- EHR integration and workflow design

**The gap between research and deployment is vast** - but now you understand what it takes to bridge it!

If this model were to be deployed in a real ICU tomorrow, we'd need to address **every single item above**. This is why clinical ML is challenging: the science is necessary but not sufficient.